In [ ]:
import os
import pandas as pd
import pickle
from string import punctuation

from nltk.tokenize import word_tokenize
from nltk.tag import pos_tag
from nltk.corpus import wordnet, stopwords
from nltk.classify import accuracy, NaiveBayesClassifier
from nltk.stem import WordNetLemmatizer
from nltk.probability import FreqDist

eng_stopwords = stopwords.words('english')
lemmatizer = WordNetLemmatizer()

MODEL_PATH = './model.pickle'
DATA_PATH = './financial_dataset.csv'

In [ ]:
def getTag(tag):
    if tag.startswith('J'):
        return 'a'
    elif tag.startswith('R'):
        return 'r'
    elif tag.startswith('V'):
        return 'v'
    else:
        return 'n'

def lemmatizing(tokens):
    lemmatized = []
    tagged = pos_tag(tokens)
    for word, tag in tagged:
        result = lemmatizer.lemmatize(word, getTag(tag))
        lemmatized.append(result)
    return lemmatized

def preprocessing(sentence):
    sentence = sentence.lower()
    tokens = word_tokenize(sentence)
    tokens = [token for token in tokens if token not in eng_stopwords]
    tokens = [token for token in tokens if token not in punctuation]
    tokens = [token for token in tokens if token.isalpha()]
    tokens = lemmatizing(tokens)
    return tokens

df = pd.read_csv(DATA_PATH)
x = df['Statement']
y = df['Sentiment']
allSentence = ' '.join(x)
allTokens = preprocessing(allSentence)
freq_dist = FreqDist(allTokens)

def extract_feature(sentence):
    featured = {}
    for word in freq_dist.keys():
        featured[word] = (word in sentence)
    return featured

feature_set = [(extract_feature(preprocessing(sentence)), sentiment)
               for sentence, sentiment in zip(x, y)
]

split = int(len(feature_set) * 0.8)
train_data = feature_set[:split]
test_data = feature_set[split:]

def trainModel():
    classifier = NaiveBayesClassifier.train(train_data)
    classifier.show_most_informative_features(7)
    acc = accuracy(classifier, test_data)
    print(f"Accuracy: {acc}")
    with open(MODEL_PATH, 'wb') as f:
        pickle.dump(classifier, f)
    return classifier

In [ ]:
def show_pos_tag(sentence):
    tokens = preprocessing(sentence)
    tagged = pos_tag(tokens)
    for word, tag in tagged:
        print(f"{word}: {tagged}")
    return tokens

def show_ant_syn(tokens):
    for token in tokens:
        antonyms = []
        synonyms = []
        for syn in wordnet.synsets(token):
            for lemma in syn.lemmas():
                synonyms.append(lemma.name())
                for ant in lemma.antonyms():
                    antonyms.append(ant.name())
        if synonyms:
            print(f"Syno : {synonyms}")
        else:
            print("No syn")
        if antonyms:
            print(f"Anto : {antonyms}")
        else:
            print("No anto")

def predictStatement(model, tokens):
    feature = extract_feature(tokens)
    predicition = model.classify(feature)
    print(f"Prediction: {predicition}")

def analyzeStatement(model, sentence):
    tokens = show_pos_tag(sentence)
    show_ant_syn(tokens)
    predictStatement(model, tokens)

In [16]:
statement = ''
while True:
    print("1. Input statement")
    print("2. Analyze Statement")
    print("3. Exit")
    choice = input(">> ")
    if choice == '3':
        break
    elif choice == '1':
        statement = input("Input statement: ")
        if len(statement.split()) < 2:
            print("Should be at least more than 1 word")
            break
        else:
            if os.path.exists(MODEL_PATH):
                with open(MODEL_PATH, 'rb') as f:
                    model = pickle.load(f)
            else:
                model = trainModel()
    elif choice == '2':
        analyzeStatement(model, statement)

1. Input statement
2. Analyze Statement
3. Exit
Most Informative Features
                decrease = True           negati : positi =     31.8 : 1.0
                    fell = True           negati : positi =     30.4 : 1.0
                     lay = True           negati : positi =     19.8 : 1.0
                   staff = True           negati : positi =     18.4 : 1.0
                  recall = True           negati : positi =     16.3 : 1.0
                   lower = True           negati : positi =     13.7 : 1.0
                    drop = True           negati : positi =     13.5 : 1.0
Accuracy: 0.7790055248618785
1. Input statement
2. Analyze Statement
3. Exit
budiono: NN
siegar: NN
No syn
No anto
No syn
No anto
Prediction: positive
1. Input statement
2. Analyze Statement
3. Exit
